# 04 — Frontier-Vergleich (Phase 5)

Eure Hand-Annotation aus Phase 2 (`annotation/meine_gold.csv`) gegen Frontier-LLM-Annotation derselben 12 Anzeigen (`annotation/frontier_gold.csv`). Output: κ-Tabelle, drei Disagreement-Beispiele, Material fürs Make-or-Buy-Memo (`memo_make_or_buy.md` im Repo-Root).

Cheatsheet: `CHEATSHEETS/frontier-llm-workflow.md`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum | _YYYY-MM-DD_ |
| Frontier-Modell | _ (z. B. `claude-opus-4-7`) |
| Prompt-Variante | _ (z. B. *v1 mit 3 Few-Shots aus IDs A, B, C*) |
| Anzahl Korrektur-Turns | _ |
| Schema-Verletzungs-Mapping | _ (Werte, die ihr gemappt habt — z. B. *möglich → teilweise*) |
| Auffälligkeiten | _ |
| Frontier-CSV | `annotation/frontier_gold.csv` |
| Eigene Gold-CSV | `annotation/meine_gold.csv` |

## Setup, Pfade und Hilfsfunktionen

Lädt mein Hand-Gold (`meine_gold.csv`) und die 12 Anzeigen-Volltexte (`annotations_auswahl.csv`), spiegelt das Schema aus `annotation/validate.py` und definiert die κ- und Normalisierungs-Helfer. **Diese Zelle zuerst ausführen** — alle weiteren Zellen bauen darauf auf.

In [ ]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 50)

# --- Pfade ---
GOLD_PATH         = Path("../annotation/meine_gold.csv")        # mein Hand-Gold (Phase 2)
AUSWAHL_PATH      = Path("../daten/annotations_auswahl.csv")    # die 12 Anzeigen mit Volltext
FRONTIER_RAW_PATH = Path("../daten/frontier_raw.json")          # HIER den Frontier-Output ablegen
FRONTIER_CSV_PATH = Path("../annotation/frontier_gold.csv")     # daraus gebaute schema-konforme CSV

# --- Schema (gespiegelt aus annotation/validate.py) ---
FIELDS = ["homeoffice", "vertragsart", "erfahrungslevel",
          "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]
CATEGORICAL_FIELDS = ["homeoffice", "vertragsart", "erfahrungslevel"]
ALLOWED = {
    "homeoffice":      {"ja", "teilweise", "nein", "remote", "nicht_genannt"},
    "vertragsart":     {"ausbildung", "festanstellung", "praktikum", "werkstudent", "sonstiges"},
    "erfahrungslevel": {"junior", "mid", "senior", "egal", "nicht_genannt"},
    "gehalt_zeitraum": {"monat", "jahr", "null"},
}

# --- Hand-Gold laden + id -> refnr vereinheitlichen + Werte trimmen ---
gold = pd.read_csv(GOLD_PATH)
if "id" in gold.columns and "refnr" not in gold.columns:
    gold = gold.rename(columns={"id": "refnr"})
for c in gold.columns:
    if gold[c].dtype == object:
        gold[c] = gold[c].astype(str).str.strip()
gold["refnr"] = gold["refnr"].astype(str).str.strip()

# --- die 12 Anzeigen-Volltexte ---
auswahl = pd.read_csv(AUSWAHL_PATH)
auswahl["refnr"] = auswahl["refnr"].astype(str).str.strip()
subset = auswahl[auswahl["refnr"].isin(gold["refnr"])].copy()

print("Gold:", gold.shape, "| Auswahl:", auswahl.shape, "| 12er-Subset:", subset.shape)
fehlt = set(gold["refnr"]) - set(subset["refnr"])
print("Anzeigen ohne Volltext:", fehlt or "keine")


# --- Hilfsfunktionen (von kappa- und Disagreement-Zellen genutzt) ---
def cohen_kappa(a, b):
    """Cohen's kappa fuer zwei parallele Label-Listen (gleiche Laenge)."""
    assert len(a) == len(b)
    n = len(a)
    if n == 0:
        return float("nan")
    po = sum(1 for x, y in zip(a, b) if x == y) / n
    ca, cb = Counter(a), Counter(b)
    pe = sum((ca[c] / n) * (cb[c] / n) for c in set(ca) | set(cb))
    if pe == 1.0:
        return float("nan")  # bei nur einer Klasse ist kappa undefiniert
    return (po - pe) / (1 - pe)


def norm_cat(v):
    """Kategoriale Werte / Zeitraum vereinheitlichen; leer/null -> 'null'."""
    s = str(v).strip().lower()
    return "null" if s in ("", "nan", "none", "null") else s


def norm_sal(v):
    """Gehalt vereinheitlichen; leer/null -> 'null', sonst Ganzzahl-String."""
    s = str(v).strip().lower()
    if s in ("", "nan", "none", "null"):
        return "null"
    try:
        return str(int(float(s)))
    except ValueError:
        return s


def norm_skills(v):
    """skills_top3 als geordnetes Tupel (Set-Match, Reihenfolge egal)."""
    return tuple(sorted({
        s.strip().lower()
        for s in str(v).split("|")
        if s.strip() and s.strip().lower() not in ("nan", "none")
    }))


## Frontier-Prompt bauen (Block 5.1)

Die nächste Zelle baut den vollständigen Eingabe-Block (Schema + 3 Few-Shot-Beispiele aus meinem Hand-Gold + alle 12 Anzeigen im Volltext) und gibt ihn fertig zum Kopieren aus. Den Prompt nicht von Hand zusammenstellen — so ist er reproduzierbar und die Anzeigen-Texte stammen garantiert aus genau den 12 Gold-Anzeigen.

In [ ]:
# ============================================================
# Frontier-Prompt bauen (ready-to-paste fuer Claude / ChatGPT)
# ============================================================
# Baut Schema + 3 Few-Shot-Beispiele (aus dem Hand-Gold) + alle 12 Anzeigen
# zu einem einzigen Eingabe-Block. Ausgabe unten einfach komplett kopieren.

# 3 Few-Shot-Beispiele aus dem Hand-Gold.
# Hinweis: diese 3 sind auch Teil der 12 Eval-Anzeigen. Dadurch ist die
# Uebereinstimmung auf genau diesen 3 etwas "antrainiert" -> im Memo erwaehnen.
FEWSHOT_IDS = [
    "15939-BB-632493-7878-9058-S",  # Praktikum
    "13509-00002110865001-S",       # Senior-Festanstellung
    "13635-7fbe73ac_JB5131141-S",   # Remote
]

SCHEMA_BLOCK = """SCHEMA (genau diese 6 Felder, nur erlaubte Werte):
- homeoffice: ja | teilweise | nein | remote | nicht_genannt
- vertragsart: ausbildung | festanstellung | praktikum | werkstudent | sonstiges
- erfahrungslevel: junior | mid | senior | egal | nicht_genannt
- gehalt_min_eur: ganze Zahl ODER null (untere Grenze, ohne Punkt/Komma)
- gehalt_zeitraum: monat | jahr | null   (null, wenn gehalt_min_eur null ist)
- skills_top3: Liste mit max. 3 konkreten technischen Skills/Tools/Methoden
  (keine Soft Skills, keine Sprachen wie Deutsch/Englisch, keine Schulabschluesse)"""

DEFINITIONS = """DEFINITIONEN (wichtige Abgrenzungen aus SCHEMA.md):
- homeoffice=remote nur bei 100% ortsunabhaengig; teilweise = hybrid / mobiles Arbeiten;
  nicht_genannt NUR, wenn die Anzeige gar nichts zum Thema sagt (keine Sicherheits-Antwort).
- erfahrungslevel: Ausbildung zaehlt immer junior; senior nur bei expliziter Senior-Nennung
  oder klarer hoher Berufserfahrung; NICHT aus Aufgabenkomplexitaet ableiten;
  nicht_genannt, wenn keine konkrete Aussage zum Level.
- gehalt_min_eur=null auch bei "nach Vereinbarung" / "attraktive Verguetung";
  bei Range ("50.000-60.000") die untere Zahl (50000).
- skills_top3: konkrete Tools/Methoden/Sprachen bevorzugen, generische Taetigkeiten
  wie "Datenanalyse" oder "Auswertung" vermeiden."""


def gold_to_json(refnr):
    """Baut aus einer Gold-Zeile das erwartete JSON-Objekt fuer ein Few-Shot-Beispiel."""
    r = gold[gold["refnr"] == refnr].iloc[0]
    sk = str(r.get("skills_top3", "")).strip()
    skills = [s.strip() for s in sk.split("|") if s.strip()][:3]
    g = str(r.get("gehalt_min_eur", "")).strip()
    gz = str(r.get("gehalt_zeitraum", "")).strip()
    return {
        "id": refnr,
        "homeoffice": r["homeoffice"],
        "vertragsart": r["vertragsart"],
        "erfahrungslevel": r["erfahrungslevel"],
        "gehalt_min_eur": int(g) if g.isdigit() else None,
        "gehalt_zeitraum": gz if gz in {"monat", "jahr"} else None,
        "skills_top3": skills,
    }


fewshot_blocks = []
for fid in FEWSHOT_IDS:
    txt = subset[subset["refnr"] == fid]["text"].iloc[0]
    fewshot_blocks.append(
        f"=== id: {fid} ===\n{txt}\n"
        f"ERWARTETES JSON:\n{json.dumps(gold_to_json(fid), ensure_ascii=False)}"
    )
fewshot_block = "\n\n".join(fewshot_blocks)

ad_blocks = [f"=== id: {r['refnr']} ===\n{r['text']}" for _, r in subset.iterrows()]
ads_block = "\n\n".join(ad_blocks)

FRONTIER_PROMPT = f"""Du bist ein praezises Information-Extraction-System fuer deutsche Stellenanzeigen.
Annotiere jede der unten stehenden Anzeigen streng nach dem festen Schema.

{SCHEMA_BLOCK}

{DEFINITIONS}

AUSGABE-REGELN:
- Gib genau EIN JSON-Array zurueck. KEIN Text davor oder danach, keine Markdown-Codebloecke.
- Genau ein Objekt pro Anzeige, gleiche Reihenfolge wie unten.
- Jedes Objekt hat das Feld "id" mit der exakten refnr aus "=== id: ... ===".
- Verwende AUSSCHLIESSLICH die erlaubten Schema-Werte. Erfinde keine neuen Werte.
- gehalt_min_eur ist eine Zahl oder null (kein String). gehalt_zeitraum ist monat, jahr oder null.

BEISPIELE (so ist das Schema gemeint):
{fewshot_block}

ANZEIGEN (insgesamt {len(subset)} - alle annotieren):
{ads_block}
"""

print(FRONTIER_PROMPT)
print("\n" + "-" * 70)
print("Zeichen:", len(FRONTIER_PROMPT), "| Few-Shot-IDs:", FEWSHOT_IDS, "| Anzeigen:", len(subset))


### Manuelle Schritte: Frontier annotieren lassen (Block 5.1)

1. **Prompt kopieren** — die Ausgabe der Zelle oben (kompletter Text) in **claude.ai (Opus 4.6+)** oder **ChatGPT (GPT-5+)** als *einen* Turn einfügen.
2. **Antwort abwarten** — das Frontier-LLM soll ein **JSON-Array** mit 12 Objekten zurückgeben (je ein `id`-Feld).
3. **Schema-Verletzungen** — falls Werte außerhalb des Schemas kommen (z. B. `homeoffice="möglich"`), entweder im selben Chat um Korrektur bitten (Korrektur-Turn im Run-Header zählen) **oder** unten in `VALUE_MAP` mappen und dokumentieren.
4. **Output speichern** — das reine JSON-Array (nur den `[...]`-Block) in eine neue Datei `daten/frontier_raw.json` kopieren.
5. **Run-Header oben ausfüllen** — Modell-Version, Datum, Prompt-Variante, Anzahl Korrektur-Turns.
6. Danach unten weiter: Lade-Zelle → κ-Zelle → Disagreement-Zelle.

> DSGVO: nur die öffentlichen Bundesagentur-Anzeigen ans Frontier geben. Auf Schul-Rechnern nach der Sitzung ausloggen.

## Frontier-Daten laden + Schema-Konformität prüfen

In [ ]:
# Frontier-Roh-Output -> annotation/frontier_gold.csv
#
# Voraussetzung: du hast das JSON-Array aus Claude/ChatGPT in
#   ../daten/frontier_raw.json
# gespeichert (reiner JSON-Text, das was der Chat ausgegeben hat).
#
# Diese Zelle baut daraus eine schema-konforme CSV mit derselben id-Spalte
# wie meine_gold.csv und prueft direkt die Schema-Konformitaet.

# Mapping-Tabelle fuer typische Schema-Verletzungen des Frontier-LLM.
# Bei Bedarf erweitern UND die Eintraege im Make-or-Buy-Memo dokumentieren.
VALUE_MAP = {
    "homeoffice":      {"möglich": "teilweise", "hybrid": "teilweise", "mobiles arbeiten": "teilweise"},
    "vertragsart":     {"freelance": "sonstiges", "freiberuflich": "sonstiges",
                        "leiharbeit": "sonstiges", "trainee": "festanstellung"},
    "erfahrungslevel": {},
}

def map_value(field, val):
    if val is None:
        return val
    key = str(val).strip().lower()
    return VALUE_MAP.get(field, {}).get(key, key)

if FRONTIER_RAW_PATH.exists():
    raw = json.loads(FRONTIER_RAW_PATH.read_text(encoding="utf-8"))
    # raw kann direkt ein Array sein oder ein Objekt, das irgendwo ein Array enthaelt
    records = raw if isinstance(raw, list) else next(v for v in raw.values() if isinstance(v, list))

    rows = []
    for obj in records:
        sk = obj.get("skills_top3") or []
        if isinstance(sk, str):
            sk = [s.strip() for s in sk.split("|") if s.strip()]
        g = obj.get("gehalt_min_eur", None)
        gz = obj.get("gehalt_zeitraum", None)
        rows.append({
            "id": str(obj.get("id", "")).strip(),
            "homeoffice":      map_value("homeoffice", obj.get("homeoffice")),
            "vertragsart":     map_value("vertragsart", obj.get("vertragsart")),
            "erfahrungslevel": map_value("erfahrungslevel", obj.get("erfahrungslevel")),
            "gehalt_min_eur":  "" if g in (None, "", "null") else int(g),
            "gehalt_zeitraum": "" if gz in (None, "", "null") else str(gz).strip().lower(),
            "skills_top3":     "|".join([str(s).strip() for s in sk][:3]),
            "notiz":           "frontier",
        })

    frontier = pd.DataFrame(rows, columns=[
        "id", "homeoffice", "vertragsart", "erfahrungslevel",
        "gehalt_min_eur", "gehalt_zeitraum", "skills_top3", "notiz",
    ])
    FRONTIER_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    frontier.to_csv(FRONTIER_CSV_PATH, index=False)
    print("Geschrieben:", FRONTIER_CSV_PATH, "| Zeilen:", len(frontier))

    # Schema-Konformitaet pruefen (gleiche Regeln wie annotation/validate.py)
    problems = []
    for _, r in frontier.iterrows():
        for f in CATEGORICAL_FIELDS:
            if r[f] not in ALLOWED[f]:
                problems.append(f"{r['id']}: {f}={r[f]!r} nicht im Schema")
        gz = str(r["gehalt_zeitraum"]).strip()
        if gz and gz not in {"monat", "jahr"}:
            problems.append(f"{r['id']}: gehalt_zeitraum={gz!r} nicht im Schema")
        g = str(r["gehalt_min_eur"]).strip()
        if g and not g.isdigit():
            problems.append(f"{r['id']}: gehalt_min_eur={g!r} ist keine ganze Zahl")
        if g and not gz:
            problems.append(f"{r['id']}: gehalt_min_eur gesetzt, aber gehalt_zeitraum leer")
        sk = [s for s in str(r["skills_top3"]).split("|") if s.strip()]
        if len(sk) > 3:
            problems.append(f"{r['id']}: skills_top3 hat {len(sk)} Eintraege (max 3)")

    # Vollstaendigkeit gegen die 12 Gold-IDs
    fehlend = set(gold["refnr"]) - set(frontier["id"].astype(str).str.strip())
    if fehlend:
        problems.append(f"Fehlende Anzeigen im Frontier-Output: {sorted(fehlend)}")

    print("\nSchema-Probleme:", problems if problems else "keine - schema-konform")
    display(frontier)
else:
    print("Noch keine Datei gefunden:", FRONTIER_RAW_PATH)
    print("Speichere zuerst den JSON-Array-Output aus Claude/ChatGPT dort und fuehre diese Zelle erneut aus.")


## κ Mensch ↔ Frontier

Hypothese-Cell *vor* dem κ-Compute: was schätzt ihr ist das Gesamt-κ? Bei welchem Feld erwartet ihr das niedrigste κ — warum?

In [ ]:
# Cohen's kappa Mensch <-> Frontier, pro Feld
#
# kappa fuer die kategorialen Felder (wie annotation/validate.py).
# Fuer gehalt_min_eur, gehalt_zeitraum und skills_top3 wird zusaetzlich
# Uebereinstimmung/Accuracy gezeigt - kappa ist dort weniger aussagekraeftig
# (Gehalt meist null, skills ist eine Menge, keine echte Kategorie).

if FRONTIER_CSV_PATH.exists():
    frontier = pd.read_csv(FRONTIER_CSV_PATH).fillna("")
    frontier["id"] = frontier["id"].astype(str).str.strip()
    fr = frontier.rename(columns={"id": "refnr"})
    for c in fr.columns:
        if fr[c].dtype == object:
            fr[c] = fr[c].astype(str).str.strip()

    m = gold.merge(fr, on="refnr", suffixes=("_gold", "_fr"))
    print("Gemeinsame Anzeigen Mensch <-> Frontier:", len(m))

    rows = []
    for field in FIELDS:
        g, p = f"{field}_gold", f"{field}_fr"
        if field == "skills_top3":
            a = [norm_skills(x) for x in m[g]]
            b = [norm_skills(x) for x in m[p]]
        elif field == "gehalt_min_eur":
            a = [norm_sal(x) for x in m[g]]
            b = [norm_sal(x) for x in m[p]]
        else:
            a = [norm_cat(x) for x in m[g]]
            b = [norm_cat(x) for x in m[p]]
        agree = sum(1 for x, y in zip(a, b) if x == y)
        n = len(a)
        rows.append({
            "feld": field,
            "kappa": round(cohen_kappa(a, b), 3),
            "uebereinstimmung": f"{agree}/{n}",
            "accuracy": round(agree / n, 3) if n else 0.0,
        })

    kappa_table = pd.DataFrame(rows)
    display(kappa_table)
    print("\nMittlere Uebereinstimmung ueber alle Felder:", round(kappa_table["accuracy"].mean(), 3))
    print("Interpretation kappa (Landis & Koch): <0 schlechter als Zufall | 0.00-0.20 schlecht |",
          "0.21-0.40 maessig | 0.41-0.60 moderat | 0.61-0.80 substanziell | 0.81-1.00 fast perfekt")
else:
    print("Noch keine annotation/frontier_gold.csv gefunden.")
    print("Erst die Lade-Zelle oben ausfuehren (frontier_raw.json -> frontier_gold.csv).")


## Drei Disagreement-Beispiele

Pro Beispiel: Anzeige-ID, euer Wert, Frontier-Wert, **wer hatte recht** — Begründung mit Bezug auf Schema-Definition oder konkrete Anzeigen-Stelle. Material fürs Make-or-Buy-Memo.

In [ ]:
# Disagreements pro Feld sichtbar machen: wo lagen Mensch und Frontier auseinander?

if FRONTIER_CSV_PATH.exists():
    info_cols = ["refnr"] + [c for c in ["titel", "firma"] if c in subset.columns]
    dm = m.merge(subset[info_cols], on="refnr", how="left")

    for field in FIELDS:
        g, p = f"{field}_gold", f"{field}_fr"
        print("=" * 90)
        print("Disagreements:", field)
        print("=" * 90)

        diffs = []
        for _, r in dm.iterrows():
            if field == "skills_top3":
                same = norm_skills(r[g]) == norm_skills(r[p])
            elif field == "gehalt_min_eur":
                same = norm_sal(r[g]) == norm_sal(r[p])
            else:
                same = norm_cat(r[g]) == norm_cat(r[p])
            if not same:
                diffs.append({
                    "refnr": r["refnr"],
                    "titel": r.get("titel", ""),
                    "mensch_gold": r[g],
                    "frontier": r[p],
                })

        if diffs:
            display(pd.DataFrame(diffs))
        else:
            print("keine Disagreements in diesem Feld")
else:
    print("Noch keine annotation/frontier_gold.csv gefunden.")
    print("Erst die Lade-Zelle und die kappa-Zelle ausfuehren.")


### Drei konkrete Disagreements interpretieren

Nach dem Frontier-Lauf hier **drei** Fälle aus den Tabellen oben ausfüllen. Pro Fall: Anzeige-ID, mein Wert, Frontier-Wert, **wer hatte recht** — mit Begründung am Schema oder an einer konkreten Textstelle. Das ist das Kernmaterial fürs Make-or-Buy-Memo.

**Disagreement 1**
- Anzeige-ID: `_`
- Feld: `_`
- Mein Wert: `_` | Frontier-Wert: `_`
- Wer hatte recht und warum: `_` (Schema-Definition oder Textstelle zitieren)
- Typ: ☐ Frontier-Halluzination ☐ meine Annotation war falsch ☐ Schema-Lücke (beide vertretbar)

**Disagreement 2**
- Anzeige-ID: `_`
- Feld: `_`
- Mein Wert: `_` | Frontier-Wert: `_`
- Wer hatte recht und warum: `_`
- Typ: ☐ Frontier-Halluzination ☐ meine Annotation war falsch ☐ Schema-Lücke

**Disagreement 3**
- Anzeige-ID: `_`
- Feld: `_`
- Mein Wert: `_` | Frontier-Wert: `_`
- Wer hatte recht und warum: `_`
- Typ: ☐ Frontier-Halluzination ☐ meine Annotation war falsch ☐ Schema-Lücke

> Faustregel fürs Memo: "Frontier sah etwas, das mein Schema nicht abbildet" (Schema-Lücke) ist ein anderer Befund als "Frontier hat halluziniert". Beide führen zu unterschiedlichen Make-or-Buy-Empfehlungen.